# CSV Preprocessing Notebook

Run this notebook to generate the processed CSV files used for modeling.

Outputs:
- `Crop_recommendation_processed.csv`
- `yield_df_processed.csv`
- `preprocessing_report.json`

In [1]:
from pathlib import Path
import json

import pandas as pd

BASE_DIR = Path.cwd()
print("Working directory:", BASE_DIR)

Working directory: c:\Users\USER\Downloads\archive


In [2]:
def clean_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    for column in cleaned.columns:
        if cleaned[column].isna().any():
            if pd.api.types.is_numeric_dtype(cleaned[column]):
                cleaned[column] = cleaned[column].fillna(cleaned[column].median())
            else:
                cleaned[column] = cleaned[column].fillna(cleaned[column].mode(dropna=True).iloc[0])
    return cleaned

def iqr_outlier_count(series: pd.Series) -> int:
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return int(((series < lower) | (series > upper)).sum())

In [3]:
crop_input = BASE_DIR / 'Crop_recommendation.csv'
crop_output = BASE_DIR / 'Crop_recommendation_processed.csv'

crop_df = pd.read_csv(crop_input)
crop_df = crop_df.drop_duplicates().reset_index(drop=True)
crop_df = clean_missing_values(crop_df)
crop_df['label'] = crop_df['label'].astype(str).str.strip()
label_map = {label: idx for idx, label in enumerate(sorted(crop_df['label'].unique()))}
crop_df['label_encoded'] = crop_df['label'].map(label_map).astype('int64')
crop_df.to_csv(crop_output, index=False)

print('Crop dataset processed')
print('Shape:', crop_df.shape)
print('Missing values:', int(crop_df.isna().sum().sum()))
print('Duplicates:', int(crop_df.duplicated().sum()))
print('Label classes:', len(label_map))

Crop dataset processed
Shape: (2200, 9)
Missing values: 0
Duplicates: 0
Label classes: 22


In [4]:
yield_input = BASE_DIR / 'yield_df.csv'
yield_output = BASE_DIR / 'yield_df_processed.csv'

yield_df = pd.read_csv(yield_input)
if 'Unnamed: 0' in yield_df.columns:
    yield_df = yield_df.drop(columns=['Unnamed: 0'])
yield_df = yield_df.drop_duplicates().reset_index(drop=True)
yield_df = clean_missing_values(yield_df)
yield_df['Area'] = yield_df['Area'].astype(str).str.strip()
yield_df['Item'] = yield_df['Item'].astype(str).str.strip()
yield_df = pd.get_dummies(yield_df, columns=['Area', 'Item'], drop_first=True, dtype='int64')
yield_df.to_csv(yield_output, index=False)

print('Yield dataset processed')
print('Shape:', yield_df.shape)
print('Missing values:', int(yield_df.isna().sum().sum()))
print('Duplicates:', int(yield_df.duplicated().sum()))

Yield dataset processed
Shape: (25932, 114)
Missing values: 0
Duplicates: 0


In [5]:
report = {
    'crop_recommendation': {
        'input': crop_input.name,
        'output': crop_output.name,
        'shape': list(crop_df.shape),
        'missing_values': int(crop_df.isna().sum().sum()),
        'duplicates': int(crop_df.duplicated().sum()),
        'label_map': label_map,
    },
    'yield_df': {
        'input': yield_input.name,
        'output': yield_output.name,
        'shape': list(yield_df.shape),
        'missing_values': int(yield_df.isna().sum().sum()),
        'duplicates': int(yield_df.duplicated().sum()),
    },
}

report_path = BASE_DIR / 'preprocessing_report.json'
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
print(json.dumps(report, indent=2))
print('Saved:', report_path.name)

{
  "crop_recommendation": {
    "input": "Crop_recommendation.csv",
    "output": "Crop_recommendation_processed.csv",
    "shape": [
      2200,
      9
    ],
    "missing_values": 0,
    "duplicates": 0,
    "label_map": {
      "apple": 0,
      "banana": 1,
      "blackgram": 2,
      "chickpea": 3,
      "coconut": 4,
      "coffee": 5,
      "cotton": 6,
      "grapes": 7,
      "jute": 8,
      "kidneybeans": 9,
      "lentil": 10,
      "maize": 11,
      "mango": 12,
      "mothbeans": 13,
      "mungbean": 14,
      "muskmelon": 15,
      "orange": 16,
      "papaya": 17,
      "pigeonpeas": 18,
      "pomegranate": 19,
      "rice": 20,
      "watermelon": 21
    }
  },
  "yield_df": {
    "input": "yield_df.csv",
    "output": "yield_df_processed.csv",
    "shape": [
      25932,
      114
    ],
    "missing_values": 0,
    "duplicates": 0
  }
}
Saved: preprocessing_report.json
